# Part 8 · Notebook 09 — The risk engine, VaR and stress tests

**Sessions:** S17 (Risk framework & the risk engine) · S18 (Market-risk measures & stress testing) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Write a risk rule and chain rules into a risk engine.
2. Compute historical VaR and CVaR, and compare them with the normal approximation.
3. Backtest VaR through crises, and see correlations change when it matters.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()

## 1. Pre-trade risk rules

Every order passes the risk engine, in the backtest as well as live. A rule looks at the order and a context and approves or rejects with a reason. Write **single-name concentration**: the position after the order, `|current $ position + qty × price|`, divided by equity, must be at most `max_weight`. Reason text: `f"{symbol} weight {w:.1%} > {max_weight:.1%}"`.

In [ ]:
ctx = {"equity": 1_000_000, "gross": 900_000, "positions": {"SPY": 150_000, "TLT": 100_000},
       "day_pnl": -12_000, "start_equity": 1_012_000}
orders = [{"symbol": "SPY", "qty": 100, "price": 500.0}, {"symbol": "SPY", "qty": 200, "price": 500.0},
          {"symbol": "TLT", "qty": -1500, "price": 90.0}, {"symbol": "GLD", "qty": 1000, "price": 220.0},
          {"symbol": "QQQ", "qty": 3000, "price": 440.0}]

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MaxPositionWeight(p.RiskRule):
    def __init__(self, max_weight):
        self.max_weight = max_weight

    def check(self, order, ctx):
        after = ...                               # ✍️ the $ position in this symbol after the order
        w = abs(after) / ctx["equity"]
        return p.RiskDecision(w <= self.max_weight, f"{order['symbol']} weight {w:.1%} > {self.max_weight:.1%}")

mine = [p.attempt(MaxPositionWeight(0.2).check, o_, ctx) for o_ in orders]
mine = p.check("MaxPositionWeight", mine, [p.MaxPositionWeight(0.2).check(o_, ctx) for o_ in orders])
[(o_["symbol"], o_["qty"], d.approved, d.reason) for o_, d in zip(orders, mine)]

## 2. The engine: a chain of rules

**Chain of Responsibility:** ask each rule in turn; the first rejection stops the chain, and the reason is prefixed with the rule's class name (`"MaxNotional: notional 1,320,000 > 250,000"`). Log every rejection as `(symbol, reason)`. If every rule approves, approve.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class RiskEngine:
    def __init__(self, rules):
        self.rules, self.log = rules, []

    def check(self, order, ctx):
        for rule in self.rules:
            d = rule.check(order, ctx)
            if not d.approved:
                ...                               # ✍️ build the prefixed reason, log it, and return a rejection
        return p.RiskDecision(True)

rules = [p.MaxNotional(250_000), p.MaxGrossExposure(1.5), p.MaxPositionWeight(0.2), p.DailyLossLimit(0.02)]
mine = p.attempt(lambda: [(d.approved, d.reason) for d in (RiskEngine(rules).check(o_, ctx) for o_ in orders)])
ref_engine = p.RiskEngine(rules)
mine = p.check("RiskEngine", mine, [(d.approved, d.reason) for d in (ref_engine.check(o_, ctx) for o_ in orders)])
pd.DataFrame(mine, index=[f"{o_['symbol']} {o_['qty']:+}" for o_ in orders], columns=["approved", "reason"])

## 3. VaR and CVaR

A 60/40-style portfolio of six synthetic assets, ten years of daily returns, with three crisis episodes in which equity correlations jump and bonds rally.

Historical **VaR** at level `α` is the `α`-quantile of the losses (`−returns`); **CVaR** (expected shortfall) is the mean of the losses at or beyond the VaR. Both as positive numbers.

In [ ]:
A = p.asset_returns()
w = np.array([0.35, 0.25, 0.25, 0.05, 0.05, 0.05])
port = pd.Series(A.to_numpy() @ w, index=A.index)
A.describe().loc[["mean", "std"]].T.assign(mean=lambda d: d["mean"] * 252, std=lambda d: d["std"] * np.sqrt(252)).round(3)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def hist_var_cvar(returns, alpha=0.99):
    losses = -np.asarray(returns, dtype=float)
    var = ...                                     # ✍️
    cvar = ...                                    # ✍️
    return float(var), float(cvar)

mine = [p.attempt(hist_var_cvar, port, a) for a in (0.95, 0.99, 0.999)]
mine = p.check("hist_var_cvar", mine, [p.hist_var_cvar(port, a) for a in (0.95, 0.99, 0.999)])
cov = A.cov().to_numpy()
pd.DataFrame({"historical VaR": [v for v, _ in mine], "historical CVaR": [c_ for _, c_ in mine],
              "normal VaR": [p.parametric_var(w, cov, a) for a in (0.95, 0.99, 0.999)]}, index=["95%", "99%", "99.9%"]).mul(100).round(2)

The normal approximation is even a little conservative at 95%, but falls behind further out: at 99.9% the historical VaR is almost twice the normal one. Fat tails and crises live there. **Component VaR** splits the (normal) VaR by asset; it adds up to the total:

In [ ]:
comp = p.component_var(w, cov)
pd.Series(comp / comp.sum(), index=A.columns, name="share of VaR").round(3).to_frame().assign(weight=w)

60% of the capital in equities carries over 90% of the risk.

## 4. VaR through crises

Estimate a 99% VaR each day from the previous 500 days and count the days it is breached (about 20 expected in 2,000 days). Then look at what the correlations did during the crises.

In [ ]:
x = port.to_numpy()
hits = {"normal": [], "historical": []}
for t in range(500, len(x)):
    win = x[t - 500:t]
    hits["normal"].append(-x[t] > 2.326 * win.std(ddof=1))
    hits["historical"].append(-x[t] > np.quantile(-win, 0.99))
print({k: int(np.sum(v)) for k, v in hits.items()}, f"breaches; expected about {0.01 * (len(x) - 500):.0f}")
crisis = np.zeros(len(A), dtype=bool)
for s in (600, 1700, 2300):
    crisis[s:s + 60] = True
pairs = [("EQ_US", "EQ_INTL"), ("EQ_US", "CMDTY"), ("EQ_US", "BOND_10Y")]
pd.DataFrame({"calm": [A[~crisis][a].corr(A[~crisis][b]) for a, b in pairs], "crisis": [A[crisis][a].corr(A[crisis][b]) for a, b in pairs]},
             index=[f"{a} / {b}" for a, b in pairs]).round(2)

Diversification shrinks exactly when it is needed: equity markets move together in a crisis. Stress tests therefore use **crisis** correlations and scenario shocks, not the calm-period covariance.

## Wrap-up

* One risk engine, rules chained, every rejection logged, in backtest and live.
* Historical VaR/CVaR, backtested; normal VaR only as a quick first look.
* Stress correlations and scenarios on top of VaR.
* Graded version: `labs/part08/week29_risk_sizing` (six rules including sector and ADV limits, VaR measures).